In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN 
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, StackingClassifier
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import LocalOutlierFactor
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from sklearn import preprocessing
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
import xgboost as xgb

In [ ]:
mydata = pd.read_csv("/content/drive/MyDrive/train.csv.zip")
testdata= pd.read_csv("/content/drive/MyDrive/test.csv.zip")

mydata["category"] = mydata.category.replace({'Strawberry_Raw': 0, 'Papaya_Ripe':1, 'Leeche_Ripe' :2, 'Orange_Ripe': 3, 'Papaya_Raw': 4, 'Mango_Ripe':5, 'Coconut_Raw':6, 'Pomengranate_Ripe':7, 'Coconut_Ripe':8, 'Guava_Ripe':9, 'Leeche_Raw':10, 'Apple_Ripe':11, 'Pomengranate_Raw':12, 'Banana_Raw':13, 'Apple_Raw':14, 'Banana_Ripe':15, 'Mango_Raw':16, 'Orange_Raw':17, 'Strawberry_Ripe':18, 'Guava_Raw':19})


kmArr= mydata.to_numpy()
tdata= testdata.to_numpy()


row, col= kmArr.shape

# To find columns with non useful data
header= np.array(mydata.columns)
lst=['ID']
for i in range(col):
  count_zeros=0
  for j in range(row):
    if kmArr[j][i]==0:
      count_zeros+=1
  if count_zeros>=800:
    lst.append(header[i])




# Deleting useless columns/features
kmArr = (mydata.drop(lst,axis=1))
tdata= (testdata.drop(lst, axis=1))

idcol= testdata['ID']



In [ ]:
# Clustering using KMeans

kmeans = KMeans(n_clusters=12,random_state=42)
cluster_labels = kmeans.fit_predict(kmArr)
kmArr= np.column_stack((kmArr, cluster_labels))


# Adding cluster as a feature to the data 
clustertest_labels = kmeans.fit_predict(tdata)
tdata= np.column_stack((tdata, clustertest_labels))


 
X_train= np.column_stack((kmArr[:, :-2], kmArr[:,-1]))
Y_train= kmArr[:,-2]
X_test= (tdata)

# normalize the data
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)


In [ ]:
# Using local outlier factor to remove outliers from the code
lof = LocalOutlierFactor(n_neighbors=13, contamination=0.1)
outlier_mask = lof.fit_predict(X_train) == -1
X_train = X_train[~outlier_mask]
Y_train = (Y_train)[~outlier_mask]
bc = BaggingClassifier(base_estimator=lof)

In [ ]:
# Logistic regression model
pred_model= LogisticRegression().fit(X_train, Y_train)


# Random forest model
rf = RandomForestClassifier(n_estimators= 240, max_depth= 26, max_features= 420, random_state=42)

rf.fit(X_train, Y_train)




# Ensembled model (Logistic regression+ Random forest)
ensemble= VotingClassifier( estimators=[('pred_model',pred_model), ('rf' ,rf)],
                            voting= 'hard')


In [ ]:
# Validation Accuracy for logistic regression model
kfold = KFold(n_splits=5,shuffle=True,random_state=42)
accuracy = cross_val_score(pred_model, X_train, Y_train, cv=10)

print("Accuracy: {:.2f} %".format(accuracy.mean() * 100))

In [ ]:
# Validation Accuracy for Random forest model
kfold = KFold(n_splits=5,shuffle=True,random_state=42)
accuracy = cross_val_score(rf, X_train, Y_train, cv=10)

print("Accuracy: {:.2f} %".format(accuracy.mean() * 100))

Accuracy: 76.87 %


In [ ]:
# Validation Accuracy for ensembled model
kfold = KFold(n_splits=5,shuffle=True,random_state=42)
accuracy = cross_val_score(ensemble, X_train, Y_train, cv=10)

print("Accuracy: {:.2f} %".format(accuracy.mean() * 100))

In [ ]:
ensemble.fit(X_train, Y_train)          # Training ensembled model
pred_cat= ensemble.predict(X_test)      # Predicted values fro ensembled model
ids= np.array(idcol)                    
res= np.column_stack((ids, pred_cat))   # Array containing id's and predicted category's numerical equivalent
res= pd.DataFrame(res)                  

headerList= ['ID', 'category']
res.to_csv("submission.csv", header= headerList, index=False)   # Creating submission.csv file

new= pd.read_csv("submission.csv")
                                        # Changing labels to category name
new["category"] = new.category.replace({0:'Strawberry_Raw', 1:'Papaya_Ripe', 2:'Leeche_Ripe', 3:'Orange_Ripe', 4:'Papaya_Raw', 5:'Mango_Ripe', 
                                           6:'Coconut_Raw', 7:'Pomengranate_Ripe', 8:'Coconut_Ripe', 9:'Guava_Ripe', 10:'Leeche_Raw', 11:'Apple_Ripe', 
                                           12:'Pomengranate_Raw', 13:'Banana_Raw', 14:'Apple_Raw', 15:'Banana_Ripe', 16:'Mango_Raw', 17:'Orange_Raw', 
                                           18:'Strawberry_Ripe', 19:'Guava_Raw'}
)


submission= pd.DataFrame(new)

headerList= ['ID', 'category']
submission.to_csv("submission.csv", header= headerList, index=False)  # Final file